In [ ]:
import os
import pandas as pd
import sqlite3

In [ ]:
# Define the path for the SQLite database
cwd = os.getcwd()
database_path = f'{cwd}/data/crash_data.db'
os.makedirs(os.path.dirname(database_path), exist_ok=True)

# Define the path for the raw crash data files downloaded
directory_path = f'{cwd}/data/reference_data'

In [ ]:
# Create/Connect to SQLite database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

In [ ]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS county_district_lut (
        OBJECTID INT,
        Cnty_Name_UC TEXT,
        Cnty_Name_PC TEXT,
        Cnty_Number INT,
        Cnty_FIPS_Number INT,
        KYTC_District_Number INT,
        D_DISTRICT TEXT
        );''')

In [ ]:
# Function to check if tables created in the database exist
def check_table_exists(db_path, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to check if the table exists
    cursor.execute('''
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
    ''', (table_name,))

    # Fetch one record
    table_exists = cursor.fetchone() is not None

    # Close the connection
    conn.close()

    return table_exists

In [ ]:
# Function to check if tables created in the database have data
def check_table_has_data(db_path, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to count the number of rows in the table
    cursor.execute(f'SELECT COUNT(*) FROM {table_name}')

    # Fetch the count
    row_count = cursor.fetchone()[0]

    # Close the connection
    conn.close()

    return row_count > 0


In [ ]:
# Check to see if tables created exist
table_name = 'county_district_lut'

if check_table_exists(database_path, table_name):
    print(f"The table '{table_name}' exists.")
else:
    print(f"The table '{table_name}' does not exist.")

In [ ]:
csv_file = directory_path + '/county_lut.csv'
file_path = os.path.join(directory_path, csv_file)
df = pd.read_csv(file_path)

In [ ]:
# Write the DataFrame to the SQLite table
df.to_sql('county_district_lut', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()
conn.close()

print(f"Data from {df} has been successfully inserted into the county_district_lut table.")
